# 🏎️ Phase 3: Advanced Feature Engineering & Strategic Logic
## Project: ApexMotors Revenue Intelligence System (v3.0)

---

### **Phase Objective**
This notebook transitions the project from raw data analysis to **Feature Engineering**. By applying automotive domain expertise, this will create high-signal variables that amplify the model's ability to distinguish between casual leads and high-intent purchasers.

**Key Deliverables:**
1. **Domain-Derived Features**: Engineering metrics like `Engagement_Density` and `High_Intent_Signal`.
2. **Lead Temperature Logic**: Converting raw days-since-contact into actionable "Hot/Warm/Cold" categories.
3. **Strategic Automation Matrix**: Mapping predicted probabilities to specific, tailor-fit marketing campaigns.
4. **Data Export**: Saving the processed feature set to the `data/processed/` directory for modeling.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler

# Initialize Root-Relative Portability
ROOT_DIR = Path.cwd()
RAW_DATA_DIR = ROOT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = ROOT_DIR / "data" / "processed"

# Ensure directory structure exists
for folder in [RAW_DATA_DIR, PROCESSED_DATA_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"✅ Infrastructure Verified. Ready to engineer features in: {PROCESSED_DATA_DIR}")

✅ Infrastructure Verified. Ready to engineer features in: /content/data/processed


### 📥 3.1 Data Requirements
**Instructions:**
Ensure the file `apex_leads_v3.csv` is located in the `data/raw/` directory. This notebook will load the raw data, apply engineering logic, and save the final version to `data/processed/` for Phase 4.

In [ ]:
# Load raw data
df = pd.read_csv(RAW_DATA_DIR / "apex_leads_v3.csv")

# 1. Engineering: Engagement Density
# Logic: Intensity of recent interest (Time spent / Days since contact)
df['Engagement_Density'] = df['App_Engagement_Mins'] / (df['Last_Contact_Days'] + 1)

# 2. Engineering: High-Intent Signal (Interaction Feature)
# Logic: Captures "Power Users" who engaged both digitally and physically
df['High_Intent_Signal'] = ((df['Web_Configurator_Status'] == 1) & (df['Test_Drive_Completed'] == 1)).astype(int)

# 3. Engineering: Lead Temperature (Categorical Bins)
# Logic: Actionable sales segments based on recency
def assign_temp(days):
    if days <= 7: return 'Hot'
    elif days <= 30: return 'Warm'
    else: return 'Cold'
df['Lead_Temperature'] = df['Last_Contact_Days'].apply(assign_temp)

# 4. Engineering: Attrition Risk (Threshold-Based)
# Logic: Identifies leads below the 60-min PCA threshold who haven't been contacted in 30+ days
df['Attrition_Risk'] = ((df['App_Engagement_Mins'] < 60) & (df['Last_Contact_Days'] > 30)).astype(int)

print("✅ Advanced Features (4) Engineered Successfully.")
df[['Engagement_Density', 'High_Intent_Signal', 'Lead_Temperature', 'Attrition_Risk']].head()

✅ Advanced Features (4) Engineered Successfully.


,Engagement_Density,High_Intent_Signal,Lead_Temperature,Attrition_Risk
0,1.122037,0,Cold,1
1,1.494465,0,Warm,0
2,1.594943,0,Warm,0
3,4.146907,0,Hot,0
4,NaN,0,Cold,0


### 🧠 3.2 Engineering Theory: Why these features matter

For "domain knowledge and creativity," we have implemented four strategic variables designed to amplify the signal within the dataset:

1. **Engagement Density**
   $$\text{Engagement Density} = \frac{\text{App Minutes}}{\text{Days Since Last Contact} + 1}$$
   * **Rationale**: This feature captures "Intent Decay." A minute of engagement spent recently is mathematically weighted higher than a minute spent months ago, allowing the model to prioritize "fresh" interest.

2. **High-Intent Signal (Non-Linear Interaction)**
   * **Rationale**: This is a binary flag identifying "Power Users" who have engaged both digitally (Web Configurator) and physically (Test Drive). This interaction feature tells the model that the combination of these behaviors is a stronger predictor than each behavior viewed in isolation.

3. **Lead Temperature (Business Segmentation)**
   * **Rationale**: This transforms raw "Days Since Contact" into actionable categories (**Hot**, **Warm**, **Cold**). This provides the sales team with immediate, intuitive bins for daily prioritization.

4. **Attrition Risk (Threshold Logic)**
   * **Rationale**: Directly derived from our Phase 2 PCA findings, this flags leads who failed to hit the **60-minute engagement tipping point** and have not been contacted in over 30 days. It serves as a trigger for automated "Save-a-Lead" marketing campaigns.

In [ ]:
# One-Hot Encoding for categorical variables
# Note: 'Attrition_Risk' and 'High_Intent_Signal' are already binary (0/1), so they stay as is.
df_final = pd.get_dummies(df, columns=['Lead_Source', 'Lead_Temperature'], drop_first=True)

# Verification Check
required_new_features = ['Engagement_Density', 'High_Intent_Signal', 'Attrition_Risk']
if all(col in df_final.columns for col in required_new_features):
    # Save to processed folder for Phase 4
    OUTPUT_FILE = PROCESSED_DATA_DIR / "apex_leads_final_v3.csv"
    df_final.to_csv(OUTPUT_FILE, index=False)

    print(f"🚀 SUCCESS: Processed dataset saved to {OUTPUT_FILE}")
    print(f"Final Feature Count: {df_final.shape[1]} (including target)")
    print(f"Included Features: {list(df_final.columns)}")
else:
    print("⚠️ Warning: Some engineered features appear to be missing. Check Cell 4.")

🚀 SUCCESS: Processed dataset saved to /content/data/processed/apex_leads_final_v3.csv
Final Feature Count: 15 (including target)
Included Features: ['Lead_ID', 'App_Engagement_Mins', 'Web_Configurator_Status', 'Test_Drive_Completed', 'Last_Contact_Days', 'Purchase', 'Engagement_Density', 'High_Intent_Signal', 'Attrition_Risk', 'Lead_Source_Social Media', 'Lead_Source_Unknown_Source_999', 'Lead_Source_Walk-in', 'Lead_Source_Web Search', 'Lead_Temperature_Hot', 'Lead_Temperature_Warm']


## 👔 3.3 Executive Strategy: Transitioning to Automation

By creating features like `Attrition_Risk`, we can now automate defensive marketing alongside offensive sales.

### **The Strategic Automation Matrix**
The model's output combined with the engineered features will trigger the following **Tailor-Fit Campaigns**:

| Predicted Probability | Category | **Automated Strategy** | **Key Signal** |
| :--- | :--- | :--- | :--- |
| **> 85%** | **VIP Priority** | **Immediate SMS Alert** to Sales Manager + Personal Invitation. | `High_Intent_Signal` |
| **60% - 84%** | **Hot Lead** | **Retargeting Ads** with specialized financing rates. | `Lead_Temp_Hot` |
| **40% - 59%** | **Nurture** | Enroll in a **5-Day Educational Email Drip**. | `Engagement_Density`|
| **< 40%** | **At Risk** | **"Save-a-Lead" Discount** or feedback survey. | `Attrition_Risk` |

> **💡 Business Insight**
> By engineering the `Attrition_Risk` feature, we are not just identifying who *will* buy but we are also identifying who we are about to *lose*. This allows ApexMotors to recover potentially lost revenue through automated re-engagement before the lead goes cold.